# Train BPR trên Google Colab để dùng cho P3

Notebook này dành cho trường hợp ZIP project bạn upload đã chứa đầy đủ:

```text
recsys-hust/
├── .git/
├── data/
├── src/
├── configs/
├── artifacts/
├── reports/
├── tests/
├── pyproject.toml
└── ...
```

Notebook **không tải lại MINDlarge**, **không chạy ingest**, và **không chạy prepare data**.

Mục tiêu cuối cùng là tạo đúng checkpoint mà P3 hiện tại đang sử dụng:

```text
artifacts/checkpoints/exp04_mind_bpr_bpr.pkl
```

Luồng chạy:

1. Upload ZIP project đầy đủ lên Colab.
2. Giải nén nguyên project, giữ cả `.git` và `data/`.
3. Cài dependencies.
4. Kiểm tra data đã processed và mapping.
5. Train BPR bằng `configs/experiment/exp04_mind_bpr.yaml`.
6. Chuyển checkpoint sang CPU để dùng trên máy Windows CPU.
7. Smoke test BPR + P3.
8. Đóng gói checkpoint và metadata để tải về máy.


## 1. Cấu hình


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

PROJECT_DIR = Path("/content/recsys-hust")
EXTRACT_DIR = Path("/content/_recsys_extract")

BPR_CONFIG = PROJECT_DIR / "configs/experiment/exp04_mind_bpr.yaml"
P3_CONFIG = PROJECT_DIR / "configs/experiment/exp07_mind_bpr_p3.yaml"

BPR_CHECKPOINT = PROJECT_DIR / "artifacts/checkpoints/exp04_mind_bpr_bpr.pkl"

print("Python:", sys.version)
print("PROJECT_DIR:", PROJECT_DIR)

## 2. Kiểm tra GPU


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: Colab hiện không có GPU. BPR vẫn chạy được bằng CPU nhưng sẽ chậm hơn.")

## 3. Upload ZIP project đầy đủ từ máy

Chạy ô dưới đây rồi chọn file ZIP project của bạn.

ZIP đã bao gồm `.git` và `data/`, vì vậy notebook sẽ dùng trực tiếp dữ liệu có sẵn.


In [ ]:
from google.colab import files

uploaded = files.upload()

zip_candidates = [
    Path("/content") / filename for filename in uploaded.keys() if filename.lower().endswith(".zip")
]

if len(zip_candidates) != 1:
    raise RuntimeError(f"Hãy upload đúng 1 file ZIP project. Hiện tìm thấy: {zip_candidates}")

PROJECT_ZIP = zip_candidates[0]

print("Đã upload:", PROJECT_ZIP)
print(f"Dung lượng: {PROJECT_ZIP.stat().st_size / (1024**3):.2f} GB")

## 4. Giải nén nguyên project

Notebook tự tìm `pyproject.toml` để xác định thư mục gốc project.

Toàn bộ `.git/`, `data/`, `artifacts/` và các file khác trong ZIP đều được giữ nguyên.


In [ ]:
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PROJECT_ZIP, "r") as zip_file:
    zip_file.extractall(EXTRACT_DIR)

pyproject_files = list(EXTRACT_DIR.rglob("pyproject.toml"))

if len(pyproject_files) == 0:
    raise FileNotFoundError("Không tìm thấy pyproject.toml trong ZIP.")

if len(pyproject_files) > 1:
    print("Tìm thấy nhiều pyproject.toml:")
    for path in pyproject_files:
        print(" -", path)
    raise RuntimeError("Không xác định được duy nhất thư mục gốc project.")

project_source = pyproject_files[0].parent

shutil.move(
    str(project_source),
    str(PROJECT_DIR),
)

shutil.rmtree(
    EXTRACT_DIR,
    ignore_errors=True,
)

os.chdir(PROJECT_DIR)

print("Project đã giải nén tại:", PROJECT_DIR)
print("Working directory:", Path.cwd())
print(".git tồn tại:", (PROJECT_DIR / ".git").exists())
print("data tồn tại:", (PROJECT_DIR / "data").exists())

## 5. Kiểm tra nhanh Git

Bước này chỉ để xác nhận `.git` trong ZIP được giữ nguyên.  
Notebook **không pull, push, checkout hoặc thay đổi branch**.


In [ ]:
git_dir = PROJECT_DIR / ".git"

if git_dir.exists():
    result = subprocess.run(
        ["git", "status", "--short", "--branch"],
        cwd=PROJECT_DIR,
        text=True,
        capture_output=True,
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)
else:
    print(
        "Không tìm thấy .git. "
        "Điều này không cản trở train BPR, "
        "nhưng khác với cấu trúc ZIP bạn mô tả."
    )

## 6. Kiểm tra đúng phiên bản project BPR + P3


In [ ]:
required_project_files = [
    PROJECT_DIR / "pyproject.toml",
    PROJECT_DIR / "src/train.py",
    PROJECT_DIR / "src/models/base.py",
    PROJECT_DIR / "src/models/bpr/__init__.py",
    PROJECT_DIR / "src/models/bpr/model.py",
    PROJECT_DIR / "src/models/bpr/recommender.py",
    BPR_CONFIG,
    PROJECT_DIR / "src/reranking/run_pipeline.py",
    P3_CONFIG,
]

missing_files = [path for path in required_project_files if not path.exists()]

if missing_files:
    print("Thiếu các file:")
    for path in missing_files:
        print(" -", path)
    raise FileNotFoundError("Project chưa đúng phiên bản cần cho BPR + P3.")

print("Cấu trúc code hợp lệ.")

for path in required_project_files:
    print("✓", path.relative_to(PROJECT_DIR))

## 7. Cài dependencies của project


In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

print("Đã cài project và dependencies.")

## 8. Kiểm tra data đã có sẵn

Notebook **không tạo lại mapping**.

Các đường dẫn quan trọng phải tồn tại:

```text
data/processed/mind_large/
├── mappings/users/
├── mappings/items/
├── train/pointwise/
└── dev/impressions/
```

Điều này đảm bảo checkpoint BPR dùng đúng `user_idx` và `item_idx` giống project local.


In [ ]:
import pyarrow.dataset as ds

processed_root = PROJECT_DIR / "data/processed/mind_large"

train_pointwise = processed_root / "train/pointwise"

dev_impressions = processed_root / "dev/impressions"

user_mapping = processed_root / "mappings/users"

item_mapping = processed_root / "mappings/items"

required_data_paths = [
    processed_root,
    train_pointwise,
    dev_impressions,
    user_mapping,
    item_mapping,
]

missing_data_paths = [path for path in required_data_paths if not path.exists()]

if missing_data_paths:
    print("Thiếu data:")
    for path in missing_data_paths:
        print(" -", path)
    raise FileNotFoundError("ZIP chưa chứa đầy đủ data/processed cần cho BPR.")

print("Đã tìm thấy đầy đủ data processed.")

## 9. Thống kê dữ liệu trước khi train


In [ ]:
num_train_rows = ds.dataset(
    str(train_pointwise),
    format="parquet",
).count_rows()

num_dev_impressions = ds.dataset(
    str(dev_impressions),
    format="parquet",
).count_rows()

num_users = ds.dataset(
    str(user_mapping),
    format="parquet",
).count_rows()

num_items = ds.dataset(
    str(item_mapping),
    format="parquet",
).count_rows()

print(
    "Train pointwise rows:",
    f"{num_train_rows:,}",
)

print(
    "Dev impressions:",
    f"{num_dev_impressions:,}",
)

print(
    "User mapping rows:",
    f"{num_users:,}",
)

print(
    "Item mapping rows:",
    f"{num_items:,}",
)

EXPECTED_NUM_ITEMS = 130_379
EXPECTED_DEV_IMPRESSIONS = 376_471

if num_items != EXPECTED_NUM_ITEMS:
    print(f"CẢNH BÁO: item mapping hiện tại không phải {EXPECTED_NUM_ITEMS:,} item.")

if num_dev_impressions != EXPECTED_DEV_IMPRESSIONS:
    print(f"CẢNH BÁO: dev impressions hiện tại không phải {EXPECTED_DEV_IMPRESSIONS:,}.")

## 10. Tính fingerprint item mapping

Fingerprint này dùng để đối chiếu mapping nếu cần.

Không tạo mapping mới và không sửa data.


In [ ]:
def calculate_item_mapping_sha256(mapping_dir):
    mapping_dataset = ds.dataset(
        str(mapping_dir),
        format="parquet",
    )

    table = mapping_dataset.to_table(
        columns=[
            "item_idx",
            "news_id",
        ]
    ).to_pandas()

    table = table.sort_values("item_idx")

    digest = hashlib.sha256()

    for row in table.itertuples(index=False):
        digest.update((f"{int(row.item_idx)}\t{row.news_id}\n").encode())

    return digest.hexdigest()


item_mapping_sha256 = calculate_item_mapping_sha256(item_mapping)

print("Item mapping SHA256:")
print(item_mapping_sha256)

## 11. Kiểm tra cấu hình BPR hiện tại

Notebook dùng nguyên:

```text
configs/experiment/exp04_mind_bpr.yaml
```

Không tự thay đổi hyperparameter.


In [ ]:
import yaml

with BPR_CONFIG.open(
    "r",
    encoding="utf-8",
) as file:
    bpr_config = yaml.safe_load(file)

print(
    json.dumps(
        bpr_config,
        indent=2,
        ensure_ascii=False,
    )
)

if bpr_config["model"]["name"] != "bpr":
    raise RuntimeError("exp04 hiện không cấu hình model BPR.")

configured_train_path = bpr_config["dataset"]["train_interactions"]

resolved_train_path = PROJECT_DIR / configured_train_path

if not resolved_train_path.exists():
    raise FileNotFoundError(f"Đường dẫn train trong YAML không tồn tại: {configured_train_path}")

print("\n✓ Cấu hình BPR hợp lệ.")

## 12. Hàm chạy lệnh và hiển thị log trực tiếp


In [ ]:
def run_streaming(command):
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )

    for line in process.stdout:
        print(
            line,
            end="",
        )

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(f"Lệnh thất bại ({return_code}): {' '.join(command)}")

## 13. Train BPR

Lệnh:

```bash
python -m src.train     --config configs/experiment/exp04_mind_bpr.yaml
```

Đây là pipeline train chung hiện tại.  
Logic train BPR thực tế nằm trong `BPRRecommender.fit()`.


In [ ]:
run_streaming(
    [
        sys.executable,
        "-m",
        "src.train",
        "--config",
        str(BPR_CONFIG.relative_to(PROJECT_DIR)),
    ]
)

if not BPR_CHECKPOINT.exists():
    raise FileNotFoundError(f"Train hoàn tất nhưng không tìm thấy checkpoint: {BPR_CHECKPOINT}")

print("\nCheckpoint đã tạo:")
print(BPR_CHECKPOINT)

print(
    "Dung lượng:",
    f"{BPR_CHECKPOINT.stat().st_size / (1024**2):.2f} MB",
)

## 14. Chuyển checkpoint sang CPU

Nếu Colab train bằng GPU, model PyTorch bên trong pickle có thể nằm trên CUDA.

Máy local của bạn là CPU-only, nên notebook chuyển model về CPU rồi save lại.


In [ ]:
from src.models.base import BaseRecommender

bpr_model = BaseRecommender.load(BPR_CHECKPOINT)

print(
    "Model class:",
    bpr_model.__class__.__name__,
)

print(
    "Device trước:",
    getattr(
        bpr_model,
        "device",
        None,
    ),
)

print(
    "Num users:",
    getattr(
        bpr_model,
        "num_users",
        None,
    ),
)

print(
    "Num items:",
    getattr(
        bpr_model,
        "num_items",
        None,
    ),
)

print(
    "Embedding dim:",
    getattr(
        bpr_model,
        "embedding_dim",
        None,
    ),
)

if (
    getattr(
        bpr_model,
        "torch_model",
        None,
    )
    is not None
):
    bpr_model.torch_model = bpr_model.torch_model.cpu()

bpr_model.device = torch.device("cpu")

bpr_model.save(BPR_CHECKPOINT)

del bpr_model

reloaded_bpr = BaseRecommender.load(BPR_CHECKPOINT)

print(
    "\nDevice sau:",
    getattr(
        reloaded_bpr,
        "device",
        None,
    ),
)

print(
    "is_fitted:",
    getattr(
        reloaded_bpr,
        "is_fitted",
        None,
    ),
)

print(
    "user_factors:",
    None if reloaded_bpr.user_factors is None else reloaded_bpr.user_factors.shape,
)

print(
    "item_factors:",
    None if reloaded_bpr.item_factors is None else reloaded_bpr.item_factors.shape,
)

if not reloaded_bpr.is_fitted:
    raise RuntimeError("Checkpoint load lại nhưng model chưa fitted.")

print("\n✓ Checkpoint CPU-portable hợp lệ.")

## 15. Smoke test BPR + P3 trên 5 impression

Bước này xác nhận checkpoint vừa train thực sự dùng được với pipeline P3 hiện tại.


In [ ]:
if not P3_CONFIG.exists():
    raise FileNotFoundError(f"Thiếu P3 config: {P3_CONFIG}")

run_streaming(
    [
        sys.executable,
        "-m",
        "src.reranking.run_pipeline",
        "--config",
        str(P3_CONFIG.relative_to(PROJECT_DIR)),
        "--sample-size",
        "5",
    ]
)

preview_path = PROJECT_DIR / "reports/reranking/exp07_mind_bpr_p3_preview.jsonl"

if not preview_path.exists():
    raise FileNotFoundError("P3 không tạo preview JSONL.")

print("\n✓ BPR checkpoint chạy được qua P3.")

print(
    "Preview:",
    preview_path,
)

## 16. Đóng gói checkpoint để tải về máy

ZIP đầu ra:

```text
bpr_checkpoint_for_p3.zip
└── artifacts/
    └── checkpoints/
        ├── exp04_mind_bpr_bpr.pkl
        └── exp04_mind_bpr_bpr_metadata.json
```

Bạn chỉ cần giải nén ZIP này vào thư mục gốc project local.


In [ ]:
checkpoint_sha256 = hashlib.sha256(BPR_CHECKPOINT.read_bytes()).hexdigest()

metadata = {
    "checkpoint_relative_path": ("artifacts/checkpoints/exp04_mind_bpr_bpr.pkl"),
    "checkpoint_sha256": (checkpoint_sha256),
    "item_mapping_sha256": (item_mapping_sha256),
    "num_catalog_items": int(num_items),
    "num_train_users": int(num_users),
    "num_dev_impressions": int(num_dev_impressions),
    "model_class": (reloaded_bpr.__class__.__name__),
    "num_model_users": int(reloaded_bpr.num_users),
    "num_model_items": int(reloaded_bpr.num_items),
    "embedding_dim": int(reloaded_bpr.embedding_dim),
    "config": bpr_config,
}

metadata_path = PROJECT_DIR / "artifacts/checkpoints/exp04_mind_bpr_bpr_metadata.json"

metadata_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with metadata_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

export_zip = Path("/content/bpr_checkpoint_for_p3.zip")

if export_zip.exists():
    export_zip.unlink()

with zipfile.ZipFile(
    export_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zip_file:
    zip_file.write(
        BPR_CHECKPOINT,
        arcname=("artifacts/checkpoints/exp04_mind_bpr_bpr.pkl"),
    )

    zip_file.write(
        metadata_path,
        arcname=("artifacts/checkpoints/exp04_mind_bpr_bpr_metadata.json"),
    )

print(
    "Đã tạo:",
    export_zip,
)

print(
    "Dung lượng ZIP:",
    f"{export_zip.stat().st_size / (1024**2):.2f} MB",
)

print(
    "Checkpoint SHA256:",
    checkpoint_sha256,
)

print(
    "Item mapping SHA256:",
    item_mapping_sha256,
)

## 17. Tải checkpoint về máy


In [ ]:
from google.colab import files

files.download("/content/bpr_checkpoint_for_p3.zip")

# Sau khi tải về máy

Giải nén:

```text
bpr_checkpoint_for_p3.zip
```

vào:

```text
C:\Users\Admin\Downloads\recsys-hust
```

Kiểm tra:

```powershell
Test-Path .\artifacts\checkpoints\exp04_mind_bpr_bpr.pkl
```

Phải trả về:

```text
True
```

Sau đó chạy:

```powershell
python -m src.reranking.run_pipeline `
    --config configs/experiment/exp07_mind_bpr_p3.yaml `
    --sample-size 1000
```

Bạn **không cần copy ngược toàn bộ data từ Colab về**, vì project local đã có data tương ứng.
